# Notebook 2 · Teach a network to recognise clothes

Part of the hands-on session **Deep Learning, Hands-on**. The page with the activities: **https://gradlabs.in/learn/deep-learning/**

**Fashion-MNIST**: 70,000 small grey pictures of 10 kinds of clothing. 60,000 to train on, 10,000 kept aside as the **test set**.

**Your goal:** more than 85% right on the test set. Then make it overfit on purpose.

Run the cells from top to bottom with **Shift + Enter**.

In [ ]:
import torch
print("PyTorch", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print("✓ GPU found:", torch.cuda.get_device_name(0))
else:
    print("No GPU. It still works, just slower.")
    print("To turn it on: Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")

## 1 · Get the pictures

In [ ]:
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

to_tensor = transforms.ToTensor()      # a picture → a 28 × 28 grid of numbers from 0 to 1
train_set = datasets.FashionMNIST("data", train=True, download=True, transform=to_tensor)
test_set = datasets.FashionMNIST("data", train=False, download=True, transform=to_tensor)

CLASSES = ["T-shirt", "Trousers", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
print(len(train_set), "training pictures,", len(test_set), "test pictures")

In [ ]:
# Look at a few
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax, i in zip(axes.flat, range(12)):
    picture, label = train_set[i]
    ax.imshow(picture[0], cmap="gray"); ax.set_title(CLASSES[label]); ax.axis("off")
plt.show()

**Batches:** the network learns from 64 pictures at a time. A `DataLoader` deals them out, shuffled.

In [ ]:
train_data = DataLoader(train_set, batch_size=64, shuffle=True)
test_data = DataLoader(test_set, batch_size=256)

## 2 · The network

784 numbers in (28 × 28 pixels) → a layer of 128 neurons → ReLU → 64 neurons → ReLU → 10 scores out, one per kind of clothing.

`nn.Flatten()` lays the 28 × 28 grid out as one long row of 784 numbers.

In [ ]:
def make_model(width=128):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, width), nn.ReLU(),
        nn.Linear(width, 64), nn.ReLU(),
        nn.Linear(64, 10),
    ).to(device)

model = make_model()
print(model)
print(sum(p.numel() for p in model.parameters()), "weights and biases to learn")

## 3 · Train

- **Loss:** `CrossEntropyLoss`, the usual loss for "pick one of several kinds". It's small when the right kind gets a high score.
- **Optimizer:** `Adam`, a smarter way to step downhill: it adjusts each weight's step size as it goes. It usually learns faster than plain SGD.
- **One round** (an *epoch*) = one pass over all 60,000 training pictures.

In [ ]:
def train_one_round(model, data, optimizer, loss_fn):
    model.train()
    for images, labels in data:
        images, labels = images.to(device), labels.to(device)
        guesses = model(images)              # 1 · guess
        loss = loss_fn(guesses, labels)      # 2 · how wrong
        optimizer.zero_grad()                # 3 · clear old gradients
        loss.backward()                      # 4 · gradients
        optimizer.step()                     # 5 · step downhill

def accuracy(model, data):
    model.eval()
    right = total = 0
    with torch.no_grad():
        for images, labels in data:
            images, labels = images.to(device), labels.to(device)
            right += (model(images).argmax(1) == labels).sum().item()   # the highest score is the guess
            total += len(labels)
    return 100 * right / total

In [ ]:
ROUNDS = 5
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

history = []
for r in range(1, ROUNDS + 1):
    train_one_round(model, train_data, optimizer, loss_fn)
    tr, te = accuracy(model, train_data), accuracy(model, test_data)
    history.append((tr, te))
    print(f"round {r}: training {tr:.1f}%, test {te:.1f}%")

Did you pass 85% on the test set? If not, ✏️ try more rounds, or `make_model(width=256)`, and run the cells again from **2 · The network**.

## 4 · Where does it go wrong?

Red titles are mistakes.

In [ ]:
images, labels = next(iter(test_data))
with torch.no_grad():
    guesses = model(images.to(device)).argmax(1).cpu()

fig, axes = plt.subplots(3, 6, figsize=(11, 6))
for ax, i in zip(axes.flat, range(18)):
    ax.imshow(images[i][0], cmap="gray"); ax.axis("off")
    ok = guesses[i] == labels[i]
    ax.set_title(f"{CLASSES[guesses[i]]}" + ("" if ok else f"\n(really {CLASSES[labels[i]]})"), color="black" if ok else "red", fontsize=9)
plt.tight_layout(); plt.show()

Shirts, T-shirts, pullovers and coats are the usual mix-ups. Would *you* always tell them apart at 28 × 28 pixels?

## 5 · ✏️ Overfit on purpose

A big network, only 1,000 training pictures, and many rounds. Watch training accuracy go to nearly 100% while test accuracy stops improving, then slips.

In [ ]:
small = DataLoader(Subset(train_set, range(1000)), batch_size=64, shuffle=True)
big = make_model(width=512)
optimizer = torch.optim.Adam(big.parameters(), lr=0.001)

curve = []
for r in range(1, 41):
    train_one_round(big, small, optimizer, loss_fn)
    curve.append((accuracy(big, small), accuracy(big, test_data)))
    if r % 5 == 0:
        print(f"round {r}: training {curve[-1][0]:.1f}%, test {curve[-1][1]:.1f}%")

plt.plot([c[0] for c in curve], label="training")
plt.plot([c[1] for c in curve], label="test")
plt.xlabel("training round"); plt.ylabel("% right"); plt.legend(); plt.show()

The gap between the two lines is overfitting: it memorised 1,000 pictures instead of learning what a shoe looks like. More data, or stopping earlier, would help.

## ✅ Done

Next: the break, then **Notebook 3** for photos, a pretrained network and an app.